In [0]:
# ============================================================
# 02_silver_transform_0604 — Silver transformation via MERGE
# Author: oakville3456
# Updated: 2026-06-06
# Branch: main
# Purpose: Clean, deduplicate and upsert Bronze → Silver
#          using Delta MERGE via Unity Catalog
# ============================================================

from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
BRONZE_TBL = "adb_retail_dev.bronze.sales"
SILVER_TBL = "adb_retail_dev.silver.sales"

# ─────────────────────────────────────────────────────────────
# READ BRONZE
# ─────────────────────────────────────────────────────────────
bronze = spark.read.table(BRONZE_TBL)

print(f"Bronze rows read: {bronze.count()}")

# ─────────────────────────────────────────────────────────────
# CLEAN + TRANSFORM
# try_to_date: returns NULL for bad dates (e.g. "2024-01-xx")
#              instead of crashing — safe for production
# ─────────────────────────────────────────────────────────────
silver_updates = (
    bronze
    .filter(F.col("price").isNotNull())
    .filter(F.col("quantity") > 0)
    .withColumn("order_date", F.try_to_date("order_date"))   # safe date parse
    .filter(F.col("order_date").isNotNull())                  # drop unparseable dates
    .withColumn("revenue", F.col("quantity") * F.col("price"))
    .dropDuplicates(["order_id", "order_date"])               # composite dedup key
)

print(f"Silver update rows (after cleaning): {silver_updates.count()}")

# ─────────────────────────────────────────────────────────────
# CREATE SILVER TABLE IF NOT EXISTS
# Safe to run on first run or any subsequent run
# ─────────────────────────────────────────────────────────────
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TBL} (
        order_id       INT,
        store_id       STRING,
        product        STRING,
        quantity       INT,
        price          DOUBLE,
        order_date     DATE,
        customer_id    STRING,
        revenue        DOUBLE,
        _ingested_at   TIMESTAMP,
        _source_file   STRING,
        _rescued_data  STRING
    )
    USING DELTA
    COMMENT 'Cleaned and deduplicated retail sales data.
     Source: Bronze Auto Loader ingestion from ADLS raw-landing.
     Deduplication key: order_id + order_date.
     Bad dates set to NULL via try_to_date.
     Updated via Delta MERGE upsert pattern.'
""")

# ─────────────────────────────────────────────────────────────
# MERGE INTO SILVER
# - match on composite key: order_id + order_date
# - existing rows → UPDATE all columns
# - new rows      → INSERT
# - idempotent: running twice produces same result
# ─────────────────────────────────────────────────────────────
(
    silver_updates.alias("u")
    .merge(
        spark.table(SILVER_TBL).alias("s"),
        "u.order_id = s.order_id AND u.order_date = s.order_date"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

# ─────────────────────────────────────────────────────────────
# DATA QUALITY CHECKS
# ─────────────────────────────────────────────────────────────
silver = spark.table(SILVER_TBL)
total      = silver.count()
dups       = silver.groupBy("order_id", "order_date").count().filter("count > 1").count()
null_dates = silver.filter(F.col("order_date").isNull()).count()
null_price = silver.filter(F.col("price").isNull()).count()
neg_price  = silver.filter(F.col("price") <= 0).count()

print("────────────────────────────────────────────")
print(f" Silver row count         : {total}")
print(f" Duplicate order+date keys: {dups}  ← must be 0")
print(f" Null order_dates         : {null_dates}  ← expected (bad source dates)")
print(f" Null prices              : {null_price}  ← must be 0")
print(f" Negative prices          : {neg_price}  ← must be 0")
print("────────────────────────────────────────────")

# Row count by date
silver.groupBy("order_date").count().orderBy("order_date").show()